In [1]:
import os, time, math, copy, random, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import pandas as pd

QUICK = False          # True -> tiny subset + few epochs, just to check that everything runs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('device:', device, '| torch', torch.__version__, '| AMP:', USE_AMP)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

def seed_all(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(0)
torch.backends.cudnn.benchmark = True

# ---- experiment sizes (full vs quick) --------------------------------------
CFG = dict(
    n_train   = 45_000 if not QUICK else 2_000,
    n_val     = 5_000  if not QUICK else 1_000,
    ep_t1     = 10 if not QUICK else 1,     # Task 1c MLP epochs
    ep_t2     = 8  if not QUICK else 1,     # Task 2c epochs per (optimizer, lr)
    ep_t3a    = 30 if not QUICK else 1,     # Task 3a epochs (5k subset)
    n_t3a     = 5_000 if not QUICK else 1_000,
    ep_t3b    = 20 if not QUICK else 1,     # Task 3b epochs per network
    depths_3b = [8, 20, 56] if not QUICK else [8, 20],
    ep_t4     = 20 if not QUICK else 1,     # Task 4 (hard limit 20)
    bs        = 128,
)
print(json.dumps(CFG, indent=1))

device: cuda | torch 2.11.0+cu130 | AMP: True
Tesla T4
{
 "n_train": 45000,
 "n_val": 5000,
 "ep_t1": 10,
 "ep_t2": 8,
 "ep_t3a": 30,
 "n_t3a": 5000,
 "ep_t3b": 20,
 "depths_3b": [
  8,
  20,
  56
 ],
 "ep_t4": 20,
 "bs": 128
}


In [ ]:
root = './data'
full_train = torchvision.datasets.CIFAR10(root, train=True, download=True)
X_all = torch.from_numpy(np.asarray(full_train.data)).permute(0, 3, 1, 2).contiguous()   # (50000,3,32,32) uint8
y_all = torch.tensor(full_train.targets, dtype=torch.long)
CLASSES = full_train.classes

# fixed, seeded 45k / 5k split
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45_000], perm[45_000:50_000]
tr_idx, va_idx = tr_idx[:CFG['n_train']], va_idx[:CFG['n_val']]

X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
X_va, y_va = X_all[va_idx], y_all[va_idx]

# per-channel mean / std from the TRAINING split only
MEAN = X_tr.float().div(255).mean(dim=(0, 2, 3))
STD  = X_tr.float().div(255).std(dim=(0, 2, 3))
print('train', tuple(X_tr.shape), 'val', tuple(X_va.shape))
print('channel mean', MEAN.numpy().round(4), 'std', STD.numpy().round(4))

X_tr, y_tr, X_va, y_va = X_tr.to(device), y_tr.to(device), X_va.to(device), y_va.to(device)
MEAN_D, STD_D = MEAN.to(device).view(1, 3, 1, 1), STD.to(device).view(1, 3, 1, 1)

def normalize(x_uint8):
    return (x_uint8.float().div_(255) - MEAN_D) / STD_D

class GPULoader:
    '''Iterates over (x, y) mini-batches that already live on the GPU.
    aug: optional function applied to the normalised float batch.'''
    def __init__(self, X, y, bs, shuffle=True, aug=None, drop_last=False):
        self.X, self.y, self.bs, self.shuffle, self.aug, self.drop_last = X, y, bs, shuffle, aug, drop_last
    def __len__(self):
        n = len(self.X)
        return n // self.bs if self.drop_last else math.ceil(n / self.bs)
    def __iter__(self):
        n = len(self.X)
        idx = torch.randperm(n, device=self.X.device) if self.shuffle else torch.arange(n, device=self.X.device)
        for i in range(0, n - (n % self.bs if self.drop_last else 0), self.bs):
            b = idx[i:i + self.bs]
            x = normalize(self.X[b])
            if self.aug is not None:
                x = self.aug(x)
            yield x, self.y[b]

# a quick look at the data
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(X_tr[i].permute(1, 2, 0).cpu().numpy()); ax.set_title(CLASSES[y_tr[i]], fontsize=8); ax.axis('off')
plt.suptitle('CIFAR-10 training samples'); plt.tight_layout(); plt.show()

 70%|██████▉   | 119M/170M [32:30<12:44, 67.3kB/s]

In [ ]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = torch.randn(in_features, out_features, device=device) * (2 / in_features) ** 0.5
        self.b = torch.zeros(out_features, device=device)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad):
        self.dW = self.x.T @ grad
        self.db = grad.sum(dim=0)
        return grad @ self.W.T


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x.clamp(min=0)

    def backward(self, grad):
        return grad * self.mask


def softmax_cross_entropy(logits, targets):
    # Numerically stable softmax
    shifted = logits - logits.max(dim=1, keepdim=True).values
    exp_scores = shifted.exp()
    probs = exp_scores / exp_scores.sum(dim=1, keepdim=True)

    n = logits.shape[0]
    loss = -torch.log(
        probs[torch.arange(n, device=device), targets] + 1e-12
    ).mean()

    grad = probs.clone()
    grad[torch.arange(n, device=device), targets] -= 1
    grad /= n

    return loss, grad

In [ ]:
class MLP:
    def __init__(self, input_size=3072, hidden_size=256, num_classes=10):
        self.fc1 = Linear(input_size, hidden_size)
        self.relu = ReLU()
        self.fc2 = Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = self.fc1.forward(x)
        x = self.relu.forward(x)
        return self.fc2.forward(x)

    def backward(self, grad):
        grad = self.fc2.backward(grad)
        grad = self.relu.backward(grad)
        self.fc1.backward(grad)

    def parameters(self):
        return [
            (self.fc1.W, self.fc1.dW),
            (self.fc1.b, self.fc1.db),
            (self.fc2.W, self.fc2.dW),
            (self.fc2.b, self.fc2.db),
        ]

In [ ]:
model = MLP().to(device) if False else MLP()

x, y = next(iter(GPULoader(X_tr, y_tr, bs=64, shuffle=True)))

logits = model.forward(x)
loss, grad = softmax_cross_entropy(logits, y)
model.backward(grad)

print("Logits shape:", logits.shape)
print("Loss:", loss.item())
print("Weight gradient shape:", model.fc1.dW.shape)